# 01. Retrieval baseline

Первый замер на golden dataset: **только retrieval**, без LLM.

Корпус: 6 статей Википедии о дифференциальных уравнениях, 132 чанка.
Индекс: Qdrant, эмбеддер `intfloat/multilingual-e5-small` (без e5-префиксов).

Метрики:
- **article hit@k** — ожидаемая статья попала в топ-k;
- **section hit@k** — ожидаемый раздел попал в топ-k;
- **recall@k** — какая доля gold-чанков попала в топ-k;
- **precision@k** — какая доля top-k является gold-чанками;
- **MRR** — средний обратный ранг первого правильного попадания.

Важно: gold-чанки теперь заданы явно в `eval/golden/gold_chunks.jsonl`
(по 1–3 на вопрос; 60 всего). Поэтому recall и precision считаются строго
по эталонным чанкам, а не по всем чанкам раздела.


In [1]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("repo:", ROOT)


repo: D:\diffeq-wiki-rag


In [2]:
from app.ingest.chunking import load_corpus
from app.rag.store import build_vectorstore
from eval.metrics import (
    evaluate_retrieval,
    load_gold_chunk_map,
    load_golden,
    save_results,
    summarize,
    summarize_by_topic,
)

items = load_golden()
corpus = load_corpus("data/corpus.jsonl")
gold_map = load_gold_chunk_map()
print(
    "вопросов:", len(items),
    "| чанков в корпусе:", len(corpus),
    "| gold-чанков:", sum(len(value) for value in gold_map.values()),
)
store = build_vectorstore()


вопросов: 30 | чанков в корпусе: 132 | gold-чанков: 60


D:\diffeq-wiki-rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3309.99it/s]

## Запуск

`top_k = 5`. Для каждого вопроса берём пять ближайших чанков и считаем
hit, recall, precision и MRR.


In [3]:
df = evaluate_retrieval(store, items, corpus, k=5, gold_map=gold_map)
summary = summarize(df)
summary


{'questions': 30,
 'article_hit@1': 0.7333,
 'section_hit@1': 0.3,
 'recall@1': 0.1056,
 'precision@1': 0.1667,
 'article_hit@3': 0.8333,
 'section_hit@3': 0.5,
 'recall@3': 0.2444,
 'precision@3': 0.1444,
 'article_hit@5': 0.9,
 'section_hit@5': 0.7,
 'recall@5': 0.3722,
 'precision@5': 0.1467,
 'mrr': 0.7872}

## По вопросам

In [4]:
df[
    [
        "id",
        "topic",
        "difficulty",
        "question",
        "article_rank",
        "section_rank",
        "gold_count",
        "relevant_retrieved",
        "recall_topk",
        "precision_topk",
        "top1_article",
        "top1_section",
    ]
]


,id,topic,difficulty,question,article_rank,section_rank,gold_count,relevant_retrieved,recall_topk,precision_topk,top1_article,top1_section
0,lipschitz-def-01,lipschitz,easy,Что такое условие Липшица для отображения межд...,1.0,1.0,2,0,0.000000,0.0,Lipschitz continuity,Definitions
1,lipschitz-example-02,lipschitz,easy,Приведи пример липшицевой функции и пример фун...,1.0,3.0,2,1,0.500000,0.2,Lipschitz continuity,Properties
2,lipschitz-props-03,lipschitz,medium,Какими свойствами обладают липшицевы функции и...,1.0,4.0,3,1,0.333333,0.2,Lipschitz continuity,One-sided Lipschitz
3,lipschitz-picard-04,lipschitz,hard,Почему условие Липшица важно для теоремы Пикар...,1.0,NaN,3,0,0.000000,0.0,Lipschitz continuity,Overview
4,linearode-general-05,linear_ode,easy,Что такое линейное дифференциальное уравнение ...,1.0,1.0,1,1,1.000000,0.2,Linear differential equation,Overview
5,linearode-homconst-06,linear_ode,medium,Как решается однородное линейное дифференциаль...,1.0,3.0,2,1,0.500000,0.2,Linear differential equation,Types of solution
6,linearode-charpoly-07,linear_ode,medium,Что такое характеристическое уравнение и как е...,1.0,4.0,2,2,1.000000,0.4,Linear differential equation,Non-homogeneous equation with constant coeffic...
7,linearode-nonhom-08,linear_ode,medium,Как устроено общее решение неоднородного линей...,5.0,NaN,2,0,0.000000,0.0,Wronskian,Application to linear differential equations
8,linearode-system-09,linear_ode,medium,Как линейное уравнение n-го порядка сводится к...,1.0,1.0,3,2,0.666667,0.4,Linear differential equation,System of linear differential equations
9,ode-def-10,ode,easy,Что такое обыкновенное дифференциальное уравне...,NaN,NaN,2,0,0.000000,0.0,Wronskian,Application to linear differential equations


## По темам

In [5]:
summarize_by_topic(df)

,topic,n,article_hit@1,article_hit@3,section_hit@3,recall@5,precision@5,mrr
0,integrating_factor,3,0.667,0.667,0.667,0.000,0.00,0.750
1,linear_ode,5,0.800,0.800,0.600,0.633,0.24,0.840
2,lipschitz,4,1.000,1.000,0.500,0.208,0.10,1.000
3,multi_hop,5,0.800,1.000,0.400,0.433,0.24,0.867
4,ode,4,0.250,0.500,0.500,0.250,0.10,0.333
5,picard,5,0.600,0.800,0.000,0.000,0.00,0.700
6,wronskian,4,1.000,1.000,1.000,1.000,0.30,1.000


## Сохранение результата

In [6]:
path = save_results(df, summary, "eval/results/retrieval_baseline.json")
print("сохранено:", path)


сохранено: eval\results\retrieval_baseline.json


## Как читать

- `article_hit@3 = 1.0` — нужная статья всегда в топ-3;
- `section_hit@3` — нужен именно раздел, а не любая часть статьи;
- `recall@5` — какую долю gold-чанков удалось поднять; для RAG это важнее
  precision, потому что LLM нужны все необходимые факты;
- `precision@5` — сколько мусора попало в контекст; влияет на стоимость,
  латентность и устойчивость ответа;
- `mrr` — насколько высоко первый правильный результат.

Если `article_hit@3` высокий, а `section_hit@3` низкий, ретривер находит
статью, но путает разделы. Именно это будем исправлять заголовками и
саммари разделов.
